In [ ]:
!pip install pygerrit2
!pip install fake_useragent

In [1]:

import pandas as pd
from pygerrit2 import GerritRestAPI, Anonymous
import urllib
import urllib.request
import json
from threading import Lock,Thread
from concurrent import futures
import time
import os
from fake_useragent import UserAgent


In [2]:
SAMPLE_DATA_PATH = 'UpdatedSampledData.xlsx'

HEADER = {'User-Agent': 'Bot_crawling_review_data_moataz_chouchen_Ets_montreal','from' : 'moataz.chouchen.1@ens.etsmtl.ca'}
URL = "https://codereview.qt-project.org"
QUERY = 'o=ALL_REVISIONS&o=ALL_FILES&o=ALL_COMMITS&o=MESSAGES&o=DETAILED_LABELS&o=DETAILED_ACCOUNTS&o=REVIEWER_UPDATES'

In [4]:
rest = GerritRestAPI(url=URL, auth=Anonymous())

In [ ]:
data = pd.read_excel('Manual Labelling.xlsx')

In [ ]:

rest.get(f'/changes/9539?{QUERY}', timeout=1000)

In [ ]:
rest.get(f'/changes/{793109}?{QUERY}', timeout=1000)


In [10]:
#helper 
def get_owner_name(change):
    return change['owner']['name']

def get_change_description(change): 
    first_revision = get_first_revision(change['revisions'])
    return first_revision['commit']['message']

def get_changed_files(change):
    first_revision = get_first_revision(change['revisions'])
    return [filename for filename in first_revision['files']]

def extract_change(id, rest_api, query): 
    return rest_api.get(f'/changes/{id}?{query}', timeout=1000)

def get_first_revision(revisions): 
    for revision_key, revision_data in revisions.items(): 
        if revision_data['_number'] == 1 :
            return revision_data



In [ ]:
labled_data = pd.read_excel(SAMPLE_DATA_PATH)
ids = labled_data['Id'].unique()
rest = GerritRestAPI(url=URL, auth=Anonymous())
res = []
for id in ids: 
    try:
        print('processing ID:', id)
        change_all_data = extract_change(id, rest, QUERY)
        new_row = {
            'Id': id, 
            'OwnerName': get_owner_name(change_all_data), 
            'Title': change_all_data['subject'], 
            'Description': get_change_description(change_all_data),
            'ChangedFiles': get_changed_files(change_all_data)
        }
        res.append(new_row)
    except:
        continue 

In [15]:
pd.DataFrame(res).to_csv('Android_sample_crawled.csv', index=False)